<a href="https://colab.research.google.com/github/FarsusDasdana/langcache_redis/blob/main/redis_semantic_caching.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the langcache library and update the Google GenAI integration
!pip install langcache
!pip install -U langchain-google-genai

In [13]:
from google.colab import userdata
from langcache import LangCache

## Connect to the LangCache instance and flush (delete) all existing entries
with LangCache(
    server_url="https://aws-us-east-1.langcache.redis.io",
    cache_id="0218ebb3d0ac4cb9bec8afd76962072c",
    api_key=userdata.get('REDIS_API_KEY'),
) as lang_cache:
    # Clear all data associated with this cache_id
    lang_cache.flush()

In [20]:
import os
import time
from google.colab import userdata

from langcache import LangCache
from langchain_google_genai import ChatGoogleGenerativeAI

# Set the API key for Google Gemini from Colab secrets
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

def cache_llm(prompt:str):
  # Initialize the Gemini model
  llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

  start_time = time.time()
  # Open a connection to LangCache
  with LangCache(
    server_url="https://aws-us-east-1.langcache.redis.io",
    cache_id="0218ebb3d0ac4cb9bec8afd76962072c",
    api_key=userdata.get('REDIS_API_KEY'),
    ) as lang_cache:

    # Step 1: Search the cache for a semantically similar prompt
    cached_response = lang_cache.search(prompt=prompt)

    if cached_response.data:
      # If found, return the cached result immediately
      print("return cached response")
      print(f"Time taken: {time.time() - start_time:.2f} seconds")
      print(cached_response.data)

    else:
      # Step 2: If not found, call the LLM
      llm_response = llm.invoke(prompt)

      # Step 3: Store the new result in the cache for future use
      res = lang_cache.set(
          prompt=prompt,
          response=llm_response.content
      )

      print("return llm generated response")
      print(f"Time taken: {time.time() - start_time:.2f} seconds")
      print(llm_response.content)

In [21]:
cache_llm("How is the weather today?")

return llm generated response
Time taken: 3.73 seconds
As an AI, I don't have real-time access to current weather conditions or your specific location.

To find out the weather today, you can:

*   **Check your phone's weather app.**
*   **Use a search engine** (like Google) and type "weather near me" or "[your city] weather."
*   **Ask a smart assistant** (Siri, Google Assistant, Alexa) "What's the weather like today?"
*   **Look at a weather website** like AccuWeather, The Weather Channel, or Weather Underground.

If you tell me your city and country, I can try to look it up for you!


In [22]:
cache_llm("How would be the weather today?")


return cached response
Time taken: 0.72 seconds
[CacheEntry(id='d1f97fcfd0366ec3519bf2ad5fc57dd2', prompt='How is the weather today?', response='As an AI, I don\'t have real-time access to current weather conditions or your specific location.\n\nTo find out the weather today, you can:\n\n*   **Check your phone\'s weather app.**\n*   **Use a search engine** (like Google) and type "weather near me" or "[your city] weather."\n*   **Ask a smart assistant** (Siri, Google Assistant, Alexa) "What\'s the weather like today?"\n*   **Look at a weather website** like AccuWeather, The Weather Channel, or Weather Underground.\n\nIf you tell me your city and country, I can try to look it up for you!', attributes={}, similarity=0.95261365, search_strategy=<SearchStrategy.SEMANTIC: 'semantic'>)]
